## Penjelasan Dan Output Soal 3

### Import SymPy dan variabel x

In [20]:
import sympy as sp

x = sp.symbols("x")

Bagian ini memanggil library SymPy untuk membuat fungsi matematika simbolik. Variabel x dibuat sebagai simbol yang nantinya digunakan dalam fungsi keanggotaan fuzzy. Semua fungsi fuzzy seperti jarak, kecepatan, dan arah akan memakai x sebagai variabel dasarnya.

### Fungsi trimf

In [21]:
def trimf(a, b, c):
    return sp.Piecewise(
        (0, x <= a),
        ((x - a) / (b - a), x <= b),
        ((c - x) / (c - b), x < c),
        (0, True)
    )

Fungsi trimf digunakan untuk membuat fungsi keanggotaan berbentuk segitiga. Fungsi ini menerima tiga parameter, yaitu batas awal, titik puncak, dan batas akhir.

Nilai keanggotaan akan bernilai 0 sebelum batas awal, naik secara bertahap sampai titik puncak, lalu turun kembali sampai batas akhir. Fungsi ini cocok untuk kategori tengah seperti “Sedang”, “Normal”, “Lurus”, “Belok_Kiri”, dan “Belok_Kanan”.

### Fungsi trapmf

In [22]:
def trapmf(a, b, c, d):
    if a == b:
        return sp.Piecewise(
            (0, x < a),
            (1, x <= c),
            ((d - x) / (d - c), x < d),
            (0, True)
        )

    if c == d:
        return sp.Piecewise(
            (0, x <= a),
            ((x - a) / (b - a), x < b),
            (1, x <= d),
            (0, True)
        )

    return sp.Piecewise(
        (0, x <= a),
        ((x - a) / (b - a), x < b),
        (1, x <= c),
        ((d - x) / (d - c), x < d),
        (0, True)
    )

Fungsi trapmf digunakan untuk membuat fungsi keanggotaan berbentuk trapesium. Fungsi ini menerima empat parameter sebagai batas bentuk trapesium.

Jika nilai awal sama, fungsi langsung bernilai penuh pada sisi kiri. Jika nilai akhir sama, fungsi dapat tetap bernilai penuh sampai sisi kanan. Kondisi ini digunakan untuk kategori ekstrem seperti “Dekat”, “Jauh”, “Lambat”, “Kencang”, “Belok_Kiri_Tajam”, dan “Belok_Kanan_Tajam”.

### Fungsi degree

In [23]:
def degree(expr, nilai):
    return float(expr.subs(x, nilai))

Fungsi degree digunakan untuk menghitung derajat keanggotaan dari suatu nilai input. Fungsi ini mengganti simbol x dengan nilai aktual, lalu mengubah hasilnya menjadi angka desimal.

Dengan fungsi ini, program bisa mengetahui seberapa besar jarak sensor kiri atau kanan masuk ke kategori “Dekat”, “Sedang”, atau “Jauh”.

### Fungsi input_range

In [24]:
def input_range(teks, minimum, maksimum):
    while True:
        try:
            nilai = float(input(teks))
            if minimum <= nilai <= maksimum:
                return nilai
            print(f"Input harus dalam range {minimum}-{maksimum}.")
        except ValueError:
            print("Input harus berupa angka.")

Fungsi input_range digunakan untuk meminta input dari pengguna dan memastikan nilainya valid.

Program hanya menerima input angka yang berada dalam rentang minimum dan maksimum. Jika nilai di luar rentang, program menampilkan pesan kesalahan. Jika input bukan angka, program juga menampilkan pesan bahwa input harus berupa angka.

### Fungsi centroid

In [25]:
def centroid(output_mf, aturan, start, end, step):
    total_titik = int((end - start) / step) + 1
    pembilang = 0.0
    penyebut = 0.0

    for i in range(total_titik):
        nilai = start + i * step
        mu = max(min(w, degree(output_mf[label], nilai)) for label, w in aturan)
        pembilang += nilai * mu
        penyebut += mu

    return pembilang / penyebut if penyebut != 0 else 0.0

Fungsi centroid digunakan untuk melakukan defuzzifikasi dengan metode centroid.

Fungsi ini menghitung nilai akhir dari output fuzzy berdasarkan aturan yang aktif. Program membagi rentang output menjadi banyak titik sesuai nilai step, lalu menghitung nilai keanggotaan pada setiap titik.

Nilai min(w, degree(...)) digunakan untuk membatasi output berdasarkan firing strength aturan. Nilai max(...) digunakan untuk mengambil gabungan terbesar dari semua aturan yang menghasilkan output. Hasil akhirnya diperoleh dari pembilang dibagi penyebut.

Jika penyebut bernilai 0, maka fungsi mengembalikan nilai 0 agar tidak terjadi pembagian dengan nol.

### Fungsi interpretasi_arah

In [26]:
def interpretasi_arah(nilai):
    if nilai < -45:
        return "Kiri Tajam"
    if nilai < -10:
        return "Kiri"
    if nilai <= 10:
        return "Lurus"
    if nilai <= 45:
        return "Kanan"
    return "Kanan Tajam"

Fungsi interpretasi_arah digunakan untuk mengubah nilai sudut arah menjadi kategori belokan.

Jika nilai sangat negatif, robot dianggap belok kiri tajam. Jika negatif sedang, robot belok kiri. Jika berada di sekitar nol, robot berjalan lurus. Jika positif, robot belok kanan. Jika sangat positif, robot belok kanan tajam.

### Fungsi interpretasi_kecepatan

In [27]:
def interpretasi_kecepatan(nilai):
    if nilai <= 35:
        return "Lambat"
    if nilai <= 70:
        return "Normal"
    return "Kencang"

Fungsi interpretasi_kecepatan digunakan untuk mengubah nilai kecepatan menjadi kategori bahasa.

Jika nilai kecepatan sampai 35, hasilnya “Lambat”. Jika sampai 70, hasilnya “Normal”. Jika lebih dari 70, hasilnya “Kencang”.

### Input jarak sensor kiri dan kanan

In [28]:
kiri_input = input_range("Masukkan jarak sensor kiri (0-200): ", 0, 200)
kanan_input = input_range("Masukkan jarak sensor kanan (0-200): ", 0, 200)

Bagian ini meminta pengguna memasukkan jarak sensor kiri dan sensor kanan.

Kedua input dibatasi dari 0 sampai 200 cm. Input ini menjadi data utama untuk menentukan arah belok dan kecepatan robot.

### Fungsi keanggotaan jarak, kecepatan, arah, dan fuzzifikasi input

In [29]:
jarak_mf = {
    "Dekat": trapmf(0, 0, 20, 60),
    "Sedang": trimf(40, 100, 160),
    "Jauh": trapmf(140, 180, 200, 200)
}

kecepatan_mf = {
    "Lambat": trapmf(0, 0, 15, 35),
    "Normal": trimf(25, 50, 75),
    "Kencang": trapmf(65, 85, 100, 100)
}

arah_mf = {
    "Belok_Kiri_Tajam": trapmf(-90, -90, -70, -30),
    "Belok_Kiri": trimf(-50, -25, 0),
    "Lurus": trimf(-15, 0, 15),
    "Belok_Kanan": trimf(0, 25, 50),
    "Belok_Kanan_Tajam": trapmf(30, 70, 90, 90)
}

kiri = {k: degree(v, kiri_input) for k, v in jarak_mf.items()}
kanan = {k: degree(v, kanan_input) for k, v in jarak_mf.items()}

Bagian ini mendefinisikan tiga kelompok fungsi keanggotaan fuzzy.

Jarak sensor memiliki kategori “Dekat”, “Sedang”, dan “Jauh”. Kecepatan memiliki kategori “Lambat”, “Normal”, dan “Kencang”. Arah belok memiliki kategori “Belok_Kiri_Tajam”, “Belok_Kiri”, “Lurus”, “Belok_Kanan”, dan “Belok_Kanan_Tajam”.

Setelah semua fungsi keanggotaan dibuat, program langsung menghitung derajat keanggotaan untuk sensor kiri dan kanan. Hasilnya disimpan dalam variabel kiri dan kanan.

### Aturan fuzzy

In [30]:
rules = [
    ("R1", "Dekat", "Dekat", "Lambat", "Lurus"),
    ("R2", "Dekat", "Sedang", "Lambat", "Belok_Kanan"),
    ("R3", "Dekat", "Jauh", "Normal", "Belok_Kanan_Tajam"),
    ("R4", "Sedang", "Dekat", "Lambat", "Belok_Kiri"),
    ("R5", "Sedang", "Sedang", "Normal", "Lurus"),
    ("R6", "Sedang", "Jauh", "Normal", "Belok_Kanan"),
    ("R7", "Jauh", "Dekat", "Normal", "Belok_Kiri_Tajam"),
    ("R8", "Jauh", "Sedang", "Normal", "Belok_Kiri"),
    ("R9", "Jauh", "Jauh", "Kencang", "Lurus")
]

Bagian ini berisi sembilan aturan fuzzy.

Setiap aturan terdiri dari kode aturan, kondisi sensor kiri, kondisi sensor kanan, output kecepatan, dan output arah belok.

Contohnya, jika sensor kiri “Dekat” dan sensor kanan “Jauh”, robot bergerak dengan kecepatan “Normal” dan arah “Belok_Kanan_Tajam”. Aturan ini menunjukkan bahwa robot harus menjauh dari sisi kiri karena objek di kiri lebih dekat.

### Firing strength, defuzzifikasi kecepatan dan arah, serta aturan aktif

In [31]:
firing = []
for kode, k, r, output_kecepatan, output_arah in rules:
    w = min(kiri[k], kanan[r])
    firing.append((kode, output_kecepatan, output_arah, w))

kecepatan = centroid(kecepatan_mf, [(ok, w) for _, ok, _, w in firing], 0, 100, 0.5)
arah = centroid(arah_mf, [(oa, w) for _, _, oa, w in firing], -90, 90, 0.5)
aktif = [kode for kode, _, _, w in firing if w > 0]

Bagian ini menghitung kekuatan aktif setiap aturan fuzzy.

Nilai w dihitung menggunakan nilai minimum antara derajat keanggotaan sensor kiri dan sensor kanan. Nilai minimum digunakan karena aturan memakai operator AND.

Setiap hasil aturan disimpan ke dalam firing, berisi kode aturan, output kecepatan, output arah, dan nilai firing strength.

Setelah itu, program melakukan defuzzifikasi untuk dua output, yaitu kecepatan dan arah. Kecepatan dihitung pada rentang 0 sampai 100 dengan step 0.5. Arah dihitung pada rentang -90 sampai 90 dengan step 0.5.

Terakhir, program membuat daftar aktif, yaitu aturan yang memiliki nilai firing strength lebih dari 0.

### Output hasil

In [32]:
print("INPUT:")
print(f"Jarak Kiri: {kiri_input:.2f} cm")
print(f"Jarak Kanan: {kanan_input:.2f} cm")

INPUT:
Jarak Kiri: 30.00 cm
Jarak Kanan: 150.00 cm


In [33]:
print("\n----------------------------------------")
print("FUZZIFIKASI")
print("----------------------------------------")
print(f"Sensor Kiri ({kiri_input:.0f} cm):")
print(f"Dekat: {kiri['Dekat']:.6f}")
print(f"Sedang: {kiri['Sedang']:.6f}")
print(f"Jauh: {kiri['Jauh']:.6f}")


----------------------------------------
FUZZIFIKASI
----------------------------------------
Sensor Kiri (30 cm):
Dekat: 0.750000
Sedang: 0.000000
Jauh: 0.000000


In [34]:
print(f"Sensor Kanan ({kanan_input:.0f} cm):")
print(f"Dekat: {kanan['Dekat']:.6f}")
print(f"Sedang: {kanan['Sedang']:.6f}")
print(f"Jauh: {kanan['Jauh']:.6f}")

Sensor Kanan (150 cm):
Dekat: 0.000000
Sedang: 0.166667
Jauh: 0.250000


In [35]:
print("\n----------------------------------------")
print("FIRING STRENGTH")
print("----------------------------------------")
for i in range(0, len(firing), 3):
    baris = firing[i:i + 3]
    print(" ".join(f"{kode}: {w:.6f}" for kode, _, _, w in baris))


----------------------------------------
FIRING STRENGTH
----------------------------------------
R1: 0.000000 R2: 0.166667 R3: 0.250000
R4: 0.000000 R5: 0.000000 R6: 0.000000
R7: 0.000000 R8: 0.000000 R9: 0.000000


In [36]:
print("\n----------------------------------------")
print("DEFUZZIFIKASI (Centroid)")
print("----------------------------------------")
print(f"Kecepatan: {kecepatan:.2f} cm/s")
print(f"Arah Belok: {arah:.2f} derajat")
print(f"Interpretasi: Belok {interpretasi_arah(arah)} dengan kecepatan {interpretasi_kecepatan(kecepatan)}")


----------------------------------------
DEFUZZIFIKASI (Centroid)
----------------------------------------
Kecepatan: 39.25 cm/s
Arah Belok: 50.28 derajat
Interpretasi: Belok Kanan Tajam dengan kecepatan Normal


In [37]:
print("\n----------------------------------------")
print("TOLAK UKUR VALIDASI")
print("----------------------------------------")
print(f"μ_Dekat_kiri({kiri_input:.2f}) = {kiri['Dekat']:.6f}")
print(f"μ_Sedang_kanan({kanan_input:.2f}) = {kanan['Sedang']:.6f}")
print(f"μ_Jauh_kanan({kanan_input:.2f}) = {kanan['Jauh']:.6f}")
print(f"Aturan aktif: {', '.join(aktif) if aktif else 'Tidak ada'}")
print(f"Robot belok kanan karena arah bernilai positif: {'Ya' if arah > 0 else 'Tidak'}")
print(f"Kecepatan moderate antara lambat dan normal: {'Ya' if 25 <= kecepatan <= 75 else 'Tidak'}")


----------------------------------------
TOLAK UKUR VALIDASI
----------------------------------------
μ_Dekat_kiri(30.00) = 0.750000
μ_Sedang_kanan(150.00) = 0.166667
μ_Jauh_kanan(150.00) = 0.250000
Aturan aktif: R2, R3
Robot belok kanan karena arah bernilai positif: Ya
Kecepatan moderate antara lambat dan normal: Ya
